# 05 — Final improvement round (v2 → v3)

Three questions, answered **without touching the test set**:

1. **Is the model underfitting?** Learning curve (more data) and complexity curve (bigger trees).
2. **Do two physics ideas help?**
   - *storm memory*: decaying 6 h / 24 h mean of Newell coupling and V·Bs over a fixed 48 h window (storms build up and recover slowly)
   - *monotonic constraints*: Kp may only rise with speed / energy input and fall with Bz
3. **Can the 80% interval be calibrated?** Conformalized quantile regression (CQR).

**Leakage protocol**
- All comparisons use the v2 HistGB settings (fixed), `TimeSeriesSplit(5)` CV inside 1998–2016, and validation 2017–2019. **Test (2020+) and the gate holdout are not used here.**
- Rule fixed in advance, applied step by step: adopt a change only if **both** CV RMSE and validation RMSE improve.
- Conformal margin: computed only from predictions on data the quantile models did not train on (out-of-fold CV years + validation years).
- This is the **last** round. `03_modeling.ipynb` is then re-run once and the model is frozen.

Runtime ≈ 20–25 min. Kernel: **Python 3.12 (tormenta)**.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src import config, train, evaluate as ev
from src.predict import load_bundle

FAST = False
CFG_MEMORY, CFG_MONO = config.USE_MEMORY_FEATURES, config.USE_MONOTONIC   # what src/config.py says now
pd.set_option("display.precision", 3)

## 1. Data and the v2 reference model

In [ ]:
X, y, persistence = train.load_xy()
masks = train.split_masks(X.index)
v2 = load_bundle(config.MODELS_DIR / "tormenta_v20261029_1003.joblib")
v2_cols = v2["features"]
params = {f"model__{k}": v for k, v in v2["params"].items()}
memory_cols = [c for c in X.columns if "_decay" in c]
print(f"v2: {len(v2_cols)} features | new storm-memory features: {memory_cols}")
print("v2 HistGB settings:", v2["params"])

## 2. Is the model underfitting?
- **Learning curve:** train on every 32nd, 16th, …, every hour of 1998–2016. If validation error still falls steeply at full data, more data would help.
- **Complexity curve:** grow the trees (7 → 255 leaves), everything else fixed. If validation error keeps falling, the model was too simple (underfitting); if it flattens or rises, model size is not the limit.

In [ ]:
lc = train.learning_curve(params, X[v2_cols], y, masks, fast=FAST)
cc = train.complexity_curve(params, X[v2_cols], y, masks, fast=FAST)
lc.to_csv(config.METRICS_DIR / "learning_curve.csv")
cc.to_csv(config.METRICS_DIR / "complexity_curve.csv")
ev.plot_diagnostic_curves(lc, cc, "15_learning_complexity")
plt.show()
display(lc)
display(cc)

## 3. Variants (fixed v2 settings, CV + validation only)

In [ ]:
variants = {}
variants["A: v2"] = train.compare_variant(params, X, y, masks, v2_cols, monotonic=False, fast=FAST)
variants["B: v2 + storm memory"] = train.compare_variant(params, X, y, masks, v2_cols + memory_cols,
                                                         monotonic=False, fast=FAST)
b, a = variants["B: v2 + storm memory"], variants["A: v2"]
ADOPT_MEMORY = bool(b["cv_rmse"] < a["cv_rmse"] and b["rmse"] < a["rmse"])
cols = v2_cols + memory_cols if ADOPT_MEMORY else v2_cols
ref = b if ADOPT_MEMORY else a

variants["C: previous winner + monotonic"] = train.compare_variant(params, X, y, masks, cols,
                                                                   monotonic=True, fast=FAST)
c = variants["C: previous winner + monotonic"]
ADOPT_MONO = bool(c["cv_rmse"] < ref["cv_rmse"] and c["rmse"] < ref["rmse"])

table = pd.DataFrame(variants).T[["cv_rmse", "rmse", "mae", "r2", "storm_rmse", "storm_mae"]]
table.to_csv(config.METRICS_DIR / "v3_variants_validation.csv")
display(table)
print(f"adopt storm memory: {ADOPT_MEMORY} | adopt monotonic constraints: {ADOPT_MONO}")

## 4. Interval calibration (conformal)
Raw coverage of the q10–q90 band on data the quantile models never saw, per calibration block, then the single margin that brings the pooled coverage to 80%. A positive margin widens the band, a negative one narrows it.

In [ ]:
config.USE_MONOTONIC = ADOPT_MONO          # this session only, so the check matches the decision
margin, cal = train.conformal_margin(params, X[cols], y, masks, fast=FAST)
config.USE_MONOTONIC = CFG_MONO
cal.to_csv(config.METRICS_DIR / "conformal_calibration.csv")
print(f"conformal margin: {margin:+.3f} Kp")
cal

## 5. Decision

In [ ]:
print(f"storm memory:  decision {ADOPT_MEMORY!s:5}  | src/config.py USE_MEMORY_FEATURES = {CFG_MEMORY}")
print(f"monotonic:     decision {ADOPT_MONO!s:5}  | src/config.py USE_MONOTONIC       = {CFG_MONO}")
if (ADOPT_MEMORY, ADOPT_MONO) == (CFG_MEMORY, CFG_MONO):
    print("\nConfig already matches → run 03_modeling.ipynb next.")
else:
    print("\nConfig does NOT match → tell Claude (or set the two flags in src/config.py), then run 03.")